In [1]:

# Import Required Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:

# Load Complete Online Retail II Dataset

file_path = Path("../dataset/online_retail_II.xlsx")

if not file_path.exists():
    raise FileNotFoundError(f"Dataset not found: {file_path.resolve()}")

# Read all Excel sheets
all_sheets = pd.read_excel(file_path, sheet_name=None)

print("Available Sheets:", list(all_sheets.keys()))

# Combine all sheets into one DataFrame
df = pd.concat(all_sheets.values(), ignore_index=True)

print("\nComplete Dataset Loaded Successfully!")
print("Total Records:", len(df))
print("Total Columns:", len(df.columns))
print("Dataset Shape:", df.shape)


Available Sheets: ['Year 2009-2010', 'Year 2010-2011']

Complete Dataset Loaded Successfully!
Total Records: 1067371
Total Columns: 8
Dataset Shape: (1067371, 8)


In [3]:

# Standardize Column Names

df.columns = df.columns.str.strip()

print("Standardized Column Names:")
print(df.columns.tolist())


Standardized Column Names:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [4]:

# Check Duplicate Records

print("Duplicates Before Cleaning:", df.duplicated().sum())

# Remove exact duplicate records
df_clean = df.drop_duplicates().copy()

print("Duplicates After Cleaning:", df_clean.duplicated().sum())
print("Records After Removing Duplicates:", len(df_clean))


Duplicates Before Cleaning: 34335
Duplicates After Cleaning: 0
Records After Removing Duplicates: 1033036


In [5]:

# Missing Value Analysis

print("Missing Values Before Cleaning:\n")

print(df_clean.isnull().sum())

print("\nTotal Missing Values:")
print(df_clean.isnull().sum().sum())


Missing Values Before Cleaning:

Invoice             0
StockCode           0
Description      4275
Quantity            0
InvoiceDate         0
Price               0
Customer ID    235151
Country             0
dtype: int64

Total Missing Values:
239426


In [6]:

# Convert Columns into Correct Data Types

# Convert numerical columns
df_clean["Quantity"] = pd.to_numeric(
    df_clean["Quantity"], errors="coerce"
)

df_clean["Price"] = pd.to_numeric(
    df_clean["Price"], errors="coerce"
)

# Convert invoice date
df_clean["InvoiceDate"] = pd.to_datetime(
    df_clean["InvoiceDate"], errors="coerce"
)

# Convert customer ID
df_clean["Customer ID"] = pd.to_numeric(
    df_clean["Customer ID"], errors="coerce"
).astype("Int64")

# Convert invoice and stock code to text
df_clean["Invoice"] = df_clean["Invoice"].astype("string")
df_clean["StockCode"] = df_clean["StockCode"].astype("string")

print("Data types converted successfully!\n")
print(df_clean.dtypes)


Data types converted successfully!

Invoice                string
StockCode              string
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             Int64
Country                   str
dtype: object


In [7]:

# Handle Missing Values

print("Records Before Cleaning:", len(df_clean))

# Remove records missing essential transaction information
df_clean = df_clean.dropna(
    subset=["InvoiceDate", "StockCode", "Quantity", "Price"]
).copy()

# Fill missing product descriptions
df_clean["Description"] = df_clean["Description"].fillna(
    "Unknown Product"
)

# Remove records with zero quantity
df_clean = df_clean[df_clean["Quantity"] != 0].copy()

# Remove records with zero or negative prices
df_clean = df_clean[df_clean["Price"] > 0].copy()

print("Records After Cleaning:", len(df_clean))
print("Remaining Missing Values:", df_clean.isnull().sum().sum())


Records Before Cleaning: 1033036
Records After Cleaning: 1027017
Remaining Missing Values: 229202


In [8]:

# Identify Cancelled and Returned Orders

df_clean["Is_Cancelled"] = (
    df_clean["Invoice"].str.startswith("C", na=False)
)

df_clean["Is_Return"] = df_clean["Quantity"] < 0

print("Cancelled Transactions:", df_clean["Is_Cancelled"].sum())

print("Returned Transactions:", df_clean["Is_Return"].sum())

print(
    "Normal Transactions:",
    (~df_clean["Is_Cancelled"] & ~df_clean["Is_Return"]).sum()
)


Cancelled Transactions: 19104
Returned Transactions: 19103
Normal Transactions: 1007913


In [9]:

# Calculate Sales Amount

df_clean["Sales"] = df_clean["Quantity"] * df_clean["Price"]

print("Sales Column Created Successfully!")

print("\nSales Summary:")
print(df_clean["Sales"].describe())


Sales Column Created Successfully!

Sales Summary:
count    1.027017e+06
mean     1.851402e+01
std      2.855476e+02
min     -1.684696e+05
25%      3.750000e+00
50%      9.950000e+00
75%      1.770000e+01
max      1.684696e+05
Name: Sales, dtype: float64


In [10]:

# Final Data Quality Check

print("FINAL DATA QUALITY REPORT")
print("=" * 50)

print("Total Records:", len(df_clean))
print("Total Columns:", len(df_clean.columns))

print("\nMissing Values:")
print(df_clean.isnull().sum())

print("\nDuplicate Records:", df_clean.duplicated().sum())

print("\nData Types:")
print(df_clean.dtypes)


FINAL DATA QUALITY REPORT
Total Records: 1027017
Total Columns: 11

Missing Values:
Invoice              0
StockCode            0
Description          0
Quantity             0
InvoiceDate          0
Price                0
Customer ID     229202
Country              0
Is_Cancelled         0
Is_Return            0
Sales                0
dtype: int64

Duplicate Records: 0

Data Types:
Invoice                 string
StockCode               string
Description             object
Quantity                 int64
InvoiceDate     datetime64[us]
Price                  float64
Customer ID              Int64
Country                    str
Is_Cancelled           boolean
Is_Return                 bool
Sales                  float64
dtype: object


In [11]:

# Display First 10 Cleaned Records

df_clean.head(10)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Is_Cancelled,Is_Return,Sales
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,False,False,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,False,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,False,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,False,False,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,False,False,30.0
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085,United Kingdom,False,False,39.6
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,False,False,30.0
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085,United Kingdom,False,False,59.5
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085,United Kingdom,False,False,30.6
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085,United Kingdom,False,False,45.0


In [12]:

# Save Cleaned Dataset

output_path = Path("../dataset/online_retail_II_cleaned.csv")

df_clean.to_csv(output_path, index=False)

print("Cleaned Dataset Saved Successfully!")
print("Saved Location:", output_path.resolve())
print("Final Dataset Shape:", df_clean.shape)


Cleaned Dataset Saved Successfully!
Saved Location: C:\Users\HP\OneDrive\Desktop\Customer-Shopping-Behavior-Analytics\dataset\online_retail_II_cleaned.csv
Final Dataset Shape: (1027017, 11)
